# Spectral Binning

TauREx forward models compute spectra on a high-resolution native wavenumber grid. For comparison with observations or for faster visualization, the native spectrum is binned down to a coarser grid.

TauREx provides several binners:
- **`SimpleBinner`** — fast nearest-neighbour binning onto a user-defined wavenumber grid.
- **`FluxBinner`** — more accurate flux-conserving binning (used automatically when binning to an observed spectrum).
- **`NativeBinner`** — pass-through (no binning).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from _shared import build_transmission_model

context = build_transmission_model(include_cia=True, include_rayleigh=True, download=False)
tm = context['tm']

print('Model ready with contributions:', [c.name for c in tm.contribution_list])

## Native-Resolution Spectrum

Calling `model()` without arguments returns the spectrum on the full native grid. This is typically tens of thousands of points.

In [ ]:
native_wn, native_rprs, _, _ = tm.model()
native_wl = 10000 / native_wn[::-1]
native_rprs = native_rprs[::-1]

print(f'Native grid: {len(native_wn)} points')
print(f'Wavelength range: {native_wl.min():.3f} to {native_wl.max():.3f} um')

## Binning with SimpleBinner

`SimpleBinner` accepts a wavenumber grid and bins the model output onto it. The `bin_model` method takes the full output tuple from `model()`.

In [ ]:
from taurex.binning import SimpleBinner

# Create a logarithmic wavelength grid (0.4 - 12 um), convert to wavenumber
bin_wl = np.logspace(np.log10(0.4), np.log10(12.0), 300)
bin_wn = np.sort(10000 / bin_wl)

binner = SimpleBinner(wngrid=bin_wn)
binned_wn, binned_rprs, _, _ = binner.bin_model(tm.model(wngrid=bin_wn))
binned_wl = 10000 / binned_wn[::-1]
binned_rprs = binned_rprs[::-1]

print(f'Binned grid: {len(binned_wn)} points')

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(native_wl, native_rprs, lw=0.3, alpha=0.5, label=f'Native ({len(native_wl)} pts)')
plt.plot(binned_wl, binned_rprs, lw=2, label=f'SimpleBinner ({len(binned_wl)} pts)')
plt.xscale('log')
plt.xlabel('Wavelength (um)')
plt.ylabel('$(R_p/R_s)^2$')
plt.title('Native vs. binned transmission spectrum')
plt.legend()
plt.grid(alpha=0.2)

## Binning with FluxBinner

`FluxBinner` performs flux-conserving binning and is more accurate than `SimpleBinner`, especially when bin widths are large relative to the spectral features.

In [ ]:
from taurex.binning import FluxBinner

flux_binner = FluxBinner(wngrid=bin_wn)
fb_wn, fb_rprs, _, _ = flux_binner.bin_model(tm.model(wngrid=bin_wn))
fb_wl = 10000 / fb_wn[::-1]
fb_rprs = fb_rprs[::-1]

plt.figure(figsize=(7, 4))
plt.plot(binned_wl, binned_rprs, lw=2, label='SimpleBinner')
plt.plot(fb_wl, fb_rprs, lw=2, ls='--', label='FluxBinner')
plt.xscale('log')
plt.xlabel('Wavelength (um)')
plt.ylabel('$(R_p/R_s)^2$')
plt.title('SimpleBinner vs. FluxBinner')
plt.legend()
plt.grid(alpha=0.2)

## Binning to an Observation Grid

When working with real data, `ObservedSpectrum` provides a `create_binner()` method that returns a `FluxBinner` matched to the observation's wavelength bins. This ensures the model is binned to exactly the same grid as the data.

```python
from taurex.data.spectrum.observed import ObservedSpectrum

obs = ObservedSpectrum('observation.dat')
obs_binner = obs.create_binner()
binned_wn, binned_rprs, _, _ = obs_binner.bin_model(tm.model(obs.wavenumberGrid))
```

The observation file is a text file with 3–4 columns: wavelength ($\mu$m), spectral data, error, and optionally bin width.